<a href="https://colab.research.google.com/github/jonaire-tate/dx702-experimental-design/blob/main/DX702_CodingQuiz_Week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DX702 Homework Reflections
Jonaire Tate | Boston University OMDS | May 2026

## Week 2: Bootstrap Simulation - Pareto Distribution

In [ ]:
import numpy as np

# Set random seed so results are reproducible
np.random.seed(42)

# Pareto distribution shape parameter
shape = 3.0

# Sample sizes to test
sample_sizes = [10, 100, 1000, 10000]

# Number of bootstrap resamples
n_bootstrap = 1000

for n in sample_sizes:
    # Draw original sample from Pareto distribution
    original_sample = np.random.pareto(shape, n)

    # Bootstrap: resample 1000 times and calculate mean each time
    bootstrap_means = []
    for _ in range(n_bootstrap):
        resample = np.random.choice(original_sample, size=n, replace=True)
        bootstrap_means.append(np.mean(resample))

    # Calculate variance of those 1000 means
    variance = np.var(bootstrap_means)
    print(f"Sample size {n}: variance of bootstrap means = {variance:.6f}")

Sample size 10: variance of bootstrap means = 0.023508
Sample size 100: variance of bootstrap means = 0.002636
Sample size 1000: variance of bootstrap means = 0.000984
Sample size 10000: variance of bootstrap means = 0.000077


### Explanation

I generated samples from a Pareto distribution and ran 1,000 bootstrap
resamples for each sample size. For each resample I calculated the mean,
then measured the variance across all 1,000 means.

As sample size increases from 10 to 10,000, the variance of the bootstrap
means gets smaller. This means larger samples produce more stable,
trustworthy estimates of the mean. With only 10 observations, one extreme
Pareto value can throw the mean off significantly. With 10,000 observations,
extreme values have much less influence.

## Week 2 Coding Quiz: Fixed Effects Regression

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# Load the dataset
df = pd.read_csv('homework_2.1.csv')
df.head()

,time,G1,G2,G3
0,0,0.882026,1.441575,0.065409
1,1,0.210079,-0.163880,0.140310
2,2,0.509369,-0.115242,0.819830
3,3,1.150447,1.014698,0.607632
4,4,0.973779,-0.046562,0.610066


In [ ]:
# Reshape from wide to long format
df_long = df.melt(id_vars='time', value_vars=['G1', 'G2', 'G3'],
                  var_name='group', value_name='outcome')

df_long.head(10)

,time,group,outcome
0,0,G1,0.882026
1,1,G1,0.210079
2,2,G1,0.509369
3,3,G1,1.150447
4,4,G1,0.973779
5,5,G1,-0.438639
6,6,G1,0.535044
7,7,G1,-0.005679
8,8,G1,0.028391
9,9,G1,0.295299


In [ ]:
# Create dummy variables for each group (fixed effects)
df_dummies = pd.get_dummies(df_long, columns=['group'], drop_first=True)

# Set up X (time + group dummies) and y (outcome)
X = df_dummies[['time', 'group_G2', 'group_G3']]
y = df_dummies['outcome']

# Run the regression
model = LinearRegression()
model.fit(X, y)

print('Slope (time coefficient):', model.coef_[0])
print('Fixed effect G2:', model.coef_[1])
print('Fixed effect G3:', model.coef_[2])
print('Intercept (G1 baseline):', model.intercept_)

Slope (time coefficient): 0.009017213376688274
Fixed effect G2: 0.5111024776067301
Fixed effect G3: 0.19047986195102576
Intercept (G1 baseline): 0.07855194562167334


## Bootstrapping Simulation

In [ ]:
# Load dataset 2
df2 = pd.read_csv('homework_2.2.csv')
df2.head()

,X,Y,Z
0,0,1.182435,-0.725820
1,0,2.714474,0.563476
2,0,0.077612,-0.435632
3,0,-0.154449,-0.104553
4,0,22.298992,-2.321273


In [ ]:
# Separate treated (X=1) and untreated (X=0)
treated = df2[df2['X'] == 1]['Y']
untreated = df2[df2['X'] == 0]['Y']

# Simple difference in means
effect = treated.mean() - untreated.mean()
print('Mean effect:', effect)

Mean effect: 2.920717264723189


In [ ]:
# Bootstrap variance of the mean effect
n_bootstrap = 1000
bootstrap_effects = []

for _ in range(n_bootstrap):
    # Resample treated and untreated separately with replacement
    treated_resample = treated.sample(len(treated), replace=True)
    untreated_resample = untreated.sample(len(untreated), replace=True)

    # Calculate effect for this resample
    effect_resample = treated_resample.mean() - untreated_resample.mean()
    bootstrap_effects.append(effect_resample)

# Calculate variance of bootstrap effects
variance = np.var(bootstrap_effects)
print('Bootstrap variance:', variance)

Bootstrap variance: 0.03144925702066011


In [ ]:
from scipy import stats

# Bootstrap linear regression coefficients
n_bootstrap = 1000
bootstrap_coefs = []

for _ in range(n_bootstrap):
    # Resample the full dataset with replacement
    resample = df2.sample(len(df2), replace=True)

    X_resample = resample[['X']]
    y_resample = resample['Y']

    # Fit linear regression
    model_b = LinearRegression()
    model_b.fit(X_resample, y_resample)
    bootstrap_coefs.append(model_b.coef_[0])

# Calculate skewness
skewness = stats.skew(bootstrap_coefs)
print('Skewness:', skewness)

Skewness: -0.018227199487085403


In [ ]:
from scipy import stats

# Bootstrap linear regression coefficients
n_bootstrap = 1000
bootstrap_coefs = []

for _ in range(n_bootstrap):
    # Resample the full dataset with replacement
    resample = df2.sample(len(df2), replace=True)

    X_resample = resample[['X']]
    y_resample = resample['Y']

    # Fit linear regression
    model_b = LinearRegression()
    model_b.fit(X_resample, y_resample)
    bootstrap_coefs.append(model_b.coef_[0])

# Calculate skewness
skewness = stats.skew(bootstrap_coefs)
print('Skewness:', skewness)

Skewness: 0.00048492821730886657
